# 7.10 · 公平性 / Fairness

> **课程定位 / Where this fits**
> 第 10 课，**Part 7 · 模型评估与优化**。
> Lesson 10, **Part 7 · Model Evaluation & Tuning**.
>
> 模型很准，但对**不同人群（性别、种族、年龄）公平吗**？在贷款、招聘、司法、保险等高风险场景，一个准确率很高的模型可能**系统性地歧视**某个群体——这既不道德也违法（GDPR、EEOC）。**算法公平性**是负责任 AI 的核心，也是近年面试的新热点。这一课讲怎么**度量**和**缓解**不公平。
> A model is accurate, but is it **fair across groups (gender, race, age)**? In lending, hiring, justice, insurance, a high-accuracy model can **systematically discriminate** against a group — unethical and illegal (GDPR, EEOC). **Algorithmic fairness** is central to responsible AI and a rising interview topic. This lesson **measures** and **mitigates** unfairness.
>
> 💼 **实战/面试视角**："怎么衡量模型公平 / 删掉敏感属性就公平了吗 / 公平和准确的权衡" 越来越常考。
> 💼 **Practical/interview angle:** "how to measure fairness / does dropping the sensitive attribute fix it / fairness-accuracy trade-off" — increasingly common.

> 💡 **面试相关 / Interview-relevant**
> - "统计平价 / 机会均等 / 均等几率 的区别"（出镜率 ★★★★）
> - "删掉敏感属性就公平了吗"（★★★★★，不！代理变量）
> - "公平性指标为什么互相冲突（不可能定理）"（★★★★）
> - "前/中/后处理 三类缓解方法"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解为什么"删掉敏感属性"不能消除偏见（代理变量）。
   Understand why "dropping the sensitive attribute" doesn't remove bias (proxies).
2. 掌握三大公平性指标：**统计平价 / 机会均等 / 均等几率**。
   Master three fairness metrics: demographic parity / equal opportunity / equalized odds.
3. 理解公平性指标**互相冲突**（不可能定理）。
   Understand fairness metrics **conflict** (impossibility theorem).
4. 用**按组调阈值**做一个后处理缓解。
   Mitigate with per-group threshold tuning (post-processing).

## 目录 / TOC
1. [先建直觉 + 代理变量陷阱 ⭐](#1)
2. [⚖️ 数据：COMPAS 风格累犯](#2)
3. [三大公平性指标 ⭐](#3)
4. [指标冲突：不可能定理 ⭐](#4)
5. [缓解：按组调阈值 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + 代理变量陷阱 ⭐ / Intuition & the Proxy Trap

最常见的错误直觉是：**"只要训练时不用'种族/性别'这一列，模型就公平了。"** 这叫"无意识公平(fairness through unawareness)"，**几乎总是失败**。
The most common wrong intuition: **"if I don't feed the model the 'race/gender' column, it's fair."** This is "fairness through unawareness", and it **almost always fails**.

原因是**代理变量(proxy)**：其它特征会"偷偷编码"敏感属性。比如邮编高度关联种族、名字关联性别、消费习惯关联年龄。删掉敏感列，模型照样能**通过代理重建它**并据此歧视——而且你还失去了**度量和纠正**偏见的能力。所以正确做法是**保留敏感属性用于审计/缓解，但确保最终决策公平**。
The reason is **proxies**: other features secretly encode the sensitive attribute. Zip code strongly correlates with race, names with gender, spending with age. Drop the sensitive column and the model **reconstructs it via proxies** and discriminates anyway — while you lose the ability to **measure and correct** the bias. The right approach: **keep the sensitive attribute for auditing/mitigation, but ensure the final decision is fair**.


<a id="2"></a>
## 2. 数据：COMPAS 风格累犯 / COMPAS-style Recidivism Data

COMPAS 是著名的累犯风险评分系统，曾被 ProPublica 揭露对黑人被告有偏。这里**合成**一个同结构数据：两个群体 A/B，真实累犯率相近，但**特征里带有与群体相关的偏差**（模拟历史数据中的系统性偏见），看模型会不会学到并放大它。
COMPAS is a famous recidivism risk system, exposed by ProPublica as biased against Black defendants. We **synthesize** a same-structure dataset: two groups A/B with similar true recidivism rates, but **features carry group-correlated bias** (mimicking systemic bias in historical data), to see if the model learns and amplifies it.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

n = 6000
group = rng.integers(0, 2, n)                          # 敏感属性: 0=群体A, 1=群体B / sensitive attr
true_risk = rng.normal(0, 1, n)                        # 真实风险因子(与群体无关)
# 历史偏见: 群体B 的记录特征被系统性地"加重"(模拟有偏的执法/记录) / biased features for group B
prior = true_risk + 0.8*group + rng.normal(0, 0.5, n)  # 这个特征对 B 偏高(代理偏见)
age = rng.normal(0, 1, n)
# 真实是否累犯: 主要由 true_risk 决定, 两群体真实率接近 / true outcome ~ true_risk only
y = (rng.random(n) < 1/(1+np.exp(-(true_risk - 0.3)))).astype(int)
X = np.c_[prior, age]                                   # 注意: 没把 group 喂进模型! / group NOT a feature

print(f"COMPAS 风格: {n} 人, 群体A {np.mean(group==0):.0%} / 群体B {np.mean(group==1):.0%}")
print(f"真实累犯率: 群体A={y[group==0].mean():.1%}, 群体B={y[group==1].mean():.1%} (相近, 说明真实风险无群体差)")
print("但特征 'prior' 对群体B 系统性偏高(历史偏见的代理) — 看模型会不会因此歧视 B")

Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, group, test_size=0.4, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)   # 训练时不含 group
proba = clf.predict_proba(Xte)[:, 1]; pred = (proba > 0.5).astype(int)
print(f"\n模型未使用 group, 但预测的'高风险'比例: 群体A={pred[gte==0].mean():.1%}, 群体B={pred[gte==1].mean():.1%}")
print("→ 群体B 被预测高风险的比例明显更高 — 偏见通过代理变量 'prior' 渗了进来(无意识公平失败)")


<a id="3"></a>
## 3. 三大公平性指标 ⭐ / Three Fairness Metrics

公平性没有唯一定义，有几个相互竞争的指标（面试要点）：
Fairness has no single definition; several competing metrics (interview points):
- **统计平价(Demographic Parity)**：各群体被预测为正(高风险/批准)的**比例相同**。$\Pr(\hat y=1\mid A)=\Pr(\hat y=1\mid B)$。常用**差异比(disparate impact)** = 两群体正率之比，法律上 <0.8 视为有问题（"80% 规则"）。
  **Demographic Parity:** equal **positive prediction rate** across groups. Often via the **disparate impact ratio** = ratio of positive rates; <0.8 is legally suspect (the "80% rule").
- **机会均等(Equal Opportunity)**：各群体在**真实为正**的人里被正确预测的比例（recall/TPR）相同。$\Pr(\hat y=1\mid y=1, A)=\Pr(\hat y=1\mid y=1, B)$。
  **Equal Opportunity:** equal **recall/TPR** across groups (among truly positive).
- **均等几率(Equalized Odds)**：各群体的 **TPR 和 FPR 都相同**（最严格）。
  **Equalized Odds:** equal **both TPR and FPR** across groups (strictest).


In [ ]:
def group_metrics(y_true, y_pred, group):
    rows = {}
    for g in [0, 1]:
        m = group == g
        yt, yp = y_true[m], y_pred[m]
        pos_rate = yp.mean()                                            # 被预测为正的比例(统计平价)
        tpr = yp[yt==1].mean() if (yt==1).any() else np.nan            # 真阳率/recall(机会均等)
        fpr = yp[yt==0].mean() if (yt==0).any() else np.nan            # 假阳率(均等几率)
        rows[f"群体{'AB'[g]}"] = {"正预测率(DP)": pos_rate, "TPR(机会均等)": tpr, "FPR": fpr}
    return pd.DataFrame(rows).T

dfm = group_metrics(yte, pred, gte)
print(dfm.round(3).to_string())
di = dfm["正预测率(DP)"].min() / dfm["正预测率(DP)"].max()      # disparate impact ratio
print(f"\n统计平价差异比 disparate impact = {di:.2f}  ({'<0.8 法律上有问题!' if di<0.8 else 'OK'})")
print(f"TPR 差距(机会均等): {abs(dfm['TPR(机会均等)'].iloc[0]-dfm['TPR(机会均等)'].iloc[1]):.3f}")
print(f"FPR 差距: {abs(dfm['FPR'].iloc[0]-dfm['FPR'].iloc[1]):.3f}")
print("→ 群体B 正预测率/FPR 偏高 = 更多 B 被'误判为高风险' (正是 COMPAS 当年被批的问题)")


<a id="4"></a>
## 4. 指标冲突：不可能定理 ⭐ / Conflicting Metrics: Impossibility

一个深刻且高频的考点：**多数公平性指标不可能同时满足**（Kleinberg 等的"不可能定理"）。除非两群体的真实基础率完全相同，否则你**无法同时**做到统计平价、校准、和均等几率——满足一个往往破坏另一个。
A deep, frequently-tested point: **most fairness metrics can't be satisfied simultaneously** (Kleinberg et al.'s "impossibility theorem"). Unless both groups have identical base rates, you **cannot jointly** achieve demographic parity, calibration, and equalized odds — satisfying one usually breaks another.

这意味着**没有技术上"完全公平"的方案**，必须**结合业务和伦理选择追求哪种公平**。例如：贷款审批可能优先"机会均等"（别漏掉任何群体里真正会还款的人），而资源分配可能优先"统计平价"。**这是一个价值判断，不是纯技术问题**——面试时讲出这一点很加分。
This means there's **no technically "perfectly fair" solution**; you must **choose which fairness to pursue based on business and ethics**. E.g. lending might prioritize equal opportunity (don't miss creditworthy people in any group), while resource allocation might prioritize demographic parity. **It's a value judgment, not a purely technical one** — saying this in an interview scores well.


In [ ]:
# 演示冲突: 强行让统计平价相等(按组调阈值使正率相等), 看 TPR/FPR 是否还能相等 / DP vs odds conflict
def positive_rate(p, thr): return (p > thr).mean()
# 为群体B 找一个阈值, 使其正预测率 = 群体A 在 0.5 阈值下的正率 / equalize positive rate
target = (proba[gte==0] > 0.5).mean()
thrB = np.quantile(proba[gte==1], 1 - target)            # B 用更高阈值以降低其正率
pred_dp = pred.copy()
pred_dp[gte==1] = (proba[gte==1] > thrB).astype(int)
dfm2 = group_metrics(yte, pred_dp, gte)
print("强行满足统计平价(两组正预测率相等)后:")
print(dfm2.round(3).to_string())
print(f"\n正预测率已基本相等(统计平价✓), 但 TPR 差距={abs(dfm2['TPR(机会均等)'].iloc[0]-dfm2['TPR(机会均等)'].iloc[1]):.3f}")
print("→ 满足了统计平价, 机会均等却被破坏 — 这就是公平指标的内在冲突(不可能定理)")


<a id="5"></a>
## 5. 缓解：按组调阈值 + 小结 ⭐ / Mitigation & Summary

缓解偏见有三类方法（按在流程中的位置分）：
Bias mitigation comes in three families (by pipeline stage):
- **前处理(pre-processing)**：改数据——重采样、重加权、变换特征去除与敏感属性的关联。
  **Pre-processing:** fix the data — resample, reweight, transform features to remove correlation with the sensitive attribute.
- **中处理(in-processing)**：改训练——在损失里加公平性约束/惩罚项。
  **In-processing:** fix training — add a fairness constraint/penalty to the loss.
- **后处理(post-processing)**：改决策——**按组用不同阈值**，使某个公平性指标达标（最简单、模型无关、最常用）。
  **Post-processing:** fix the decision — **use group-specific thresholds** to meet a chosen fairness metric (simplest, model-agnostic, most common).

下面用后处理：为两组各选阈值，使 **TPR 尽量相等（机会均等）**，看代价（整体精度略降——这就是公平-准确权衡）。
Below, post-processing: pick per-group thresholds to **equalize TPR (equal opportunity)**, and see the cost (slight accuracy drop — the fairness-accuracy trade-off).


In [ ]:
from sklearn.metrics import accuracy_score

# 为每组找阈值, 使两组 TPR 尽量接近一个目标(机会均等) / per-group thresholds to equalize TPR
def tpr_at(p, yt, thr):
    pos = yt == 1
    return ((p > thr) & pos).sum() / pos.sum()

target_tpr = 0.70
pred_eo = np.zeros_like(pred)
chosen = {}
for g in [0, 1]:
    m = gte == g
    # 在候选阈值里找使该组 TPR 最接近 target 的 / threshold giving TPR closest to target
    cand = np.linspace(0.05, 0.95, 91)
    thr = cand[np.argmin([abs(tpr_at(proba[m], yte[m], t) - target_tpr) for t in cand])]
    chosen[g] = thr
    pred_eo[m] = (proba[m] > thr).astype(int)

dfm3 = group_metrics(yte, pred_eo, gte)
print(f"按组阈值: 群体A={chosen[0]:.2f}, 群体B={chosen[1]:.2f} (B 用更高阈值纠偏)")
print(dfm3.round(3).to_string())
print(f"\nTPR 差距: 缓解前 {abs(group_metrics(yte,pred,gte)['TPR(机会均等)'].diff().iloc[-1]):.3f} "
      f"→ 缓解后 {abs(dfm3['TPR(机会均等)'].diff().iloc[-1]):.3f} (机会均等改善)")
print(f"整体准确率: 缓解前 {accuracy_score(yte, pred):.3f} → 缓解后 {accuracy_score(yte, pred_eo):.3f} (略降=公平的代价)")
print("\n💡 公平-准确权衡: 提升公平通常牺牲一点整体精度 — 这是价值判断, 不是纯技术问题")


```
代理变量陷阱: 删掉敏感属性不能消除偏见(其它特征会编码它); 应保留以审计/缓解
三大指标: 统计平价(各组正预测率相等, 差异比<0.8 法律有问题) /
          机会均等(各组 TPR/recall 相等) / 均等几率(TPR 和 FPR 都相等, 最严)
不可能定理: 基础率不同时, 多数公平指标无法同时满足 → 必须按业务/伦理选一种
缓解三类: 前处理(改数据) / 中处理(改损失) / 后处理(按组调阈值, 最常用)
公平-准确权衡: 提升公平通常牺牲一点精度; 这是价值判断
```

### 💡 面试速查 / Interview cheat-sheet
1. **删掉敏感属性≠公平**(代理变量会重建它); 保留它用于审计/缓解。
   Dropping the sensitive attribute ≠ fair (proxies reconstruct it); keep it for auditing/mitigation.
2. **统计平价/机会均等/均等几率**: 正率相等 / TPR相等 / TPR+FPR都相等。
   Demographic parity / equal opportunity / equalized odds: equal positive rate / TPR / TPR+FPR.
3. **不可能定理**: 基础率不同时公平指标无法同时满足。
   Impossibility theorem: with different base rates, metrics can't all hold.
4. **缓解三类**: 前处理(数据)/中处理(损失)/后处理(按组阈值)。
   Three mitigation families: pre-/in-/post-processing.
5. **公平-准确权衡**, 选哪种公平是价值判断, 非纯技术。
   Fairness-accuracy trade-off; choosing a fairness notion is a value judgment.

### 下一节 / Next
**7.11 鲁棒性与对抗样本**——模型在干净数据上很准, 但对**精心设计的微小扰动**可能完全失效。FGSM 对抗攻击与防御直觉。
**7.11 Robustness & Adversarial Examples** — models accurate on clean data can fail completely under tiny crafted perturbations. The FGSM attack and defense intuition.
